# Gaussian Elimination

Run from the repository folder. `gaussian` modifies floating-point arrays in place, so pass copies to keep the original A and b for checking the residual. Its output stores U on and above the diagonal and the L multipliers below it. Backward substitution uses only U.


In [28]:
import sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start this notebook inside the repository")
sys.path.insert(0, str(ROOT))


In [23]:
from direct_methods.gaussian_elim import gaussian, gaussian_with_partial_pivoting
from common import loader, data_path
import numpy as np


def backward_substitution(LU, c):
    n = len(c)
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (c[i] - LU[i, i+1:] @ x[i+1:]) / LU[i, i]
    return x


## 3x3

The expected solution is `[1, 1, 2]`.


In [24]:
A = np.array([[2, 1, 1],
              [4, 3, 3],
              [8, 7, 9]], dtype=float)
b = np.array([5, 13, 33], dtype=float)

LU, c = gaussian(A.copy(), b.copy())
x = backward_substitution(LU, c)
print("x:", x)
print("Relative residual:", np.linalg.norm(A @ x - b) / np.linalg.norm(b))


x: [1. 1. 2.]
Relative residual: 0.0


## SuiteSparse: bcsstk09


In [25]:
A_raw = loader("suitesparse", path=data_path("bcsstk09"))
b_raw = A_raw @ np.ones(len(A_raw))
print("Matrix shape:", A_raw.shape)
x_true = np.ones(len(A_raw))

try:
    LU, c = gaussian(A_raw.copy(), b_raw.copy())
    x = backward_substitution(LU, c)
    print("Relative residual:", np.linalg.norm(A_raw @ x - b_raw) / np.linalg.norm(b_raw))
    print("Relative solution error:", np.linalg.norm(x - x_true) / np.linalg.norm(x_true))
except ZeroDivisionError:
    print("Zero pivot")


Matrix shape: (1083, 1083)
Relative residual: 8.648413581025025e-16
Relative solution error: 1.3617573275410154e-13


In [26]:
A = loader("suitesparse", path=data_path("west1505"))
b = A @ np.ones(len(A))

x_true = np.ones(len(A))

LU, c = gaussian_with_partial_pivoting(A.copy(), b.copy())
x = backward_substitution(LU, c)
print("Relative residual:", np.linalg.norm(A @ x - b) / np.linalg.norm(b))
print("Relative solution error:", np.linalg.norm(x - x_true) / np.linalg.norm(x_true))


Relative residual: 1.0244779905844174e-16
Relative solution error: 1.174242139134094e-09


In [27]:
# A = loader("suitesparse", path=data_path("bcsstk17"))
# b = A @ np.ones(len(A))
# print(A.shape)
# print(A.dtype)
# print(np.count_nonzero(A))
# print(np.min(np.abs(np.diag(A))))
# print(np.all(np.isfinite(A)))
# x_true = np.ones(len(A))

# LU, c = gaussian_with_partial_pivoting(A.copy(), b.copy())
# x = backward_substitution(LU, c)
# print("Relative residual:", np.linalg.norm(A @ x - b) / np.linalg.norm(b))
# print("Relative solution error:", np.linalg.norm(x - x_true) / np.linalg.norm(x_true))